# EX ANN - Классификация Iris с искусственной нейронной сетью

В этом ноутбуке ты с нуля строишь простую нейросеть, которая должна достичь не менее 95% точности (`accuracy`) при распознавании трёх видов ирисов.
- Датасет: `iris.data`
- Входы: sepal length, sepal width, petal length, petal width
- Выход: три класса, оцениваются через softmax-активацию


**Рекомендации:**
- Используй Pandas для чтения данных и инструменты Numpy/Scikit-Learn/TensorFlow для построения модели.
- Держи функции «чистыми»: не изменяй глобальные переменные, кроме констант.
- Если что-то не работает, печатай промежуточные результаты (`head`, `value_counts`, `shape`) для быстрой диагностики.

**Google Drive:** загрузи `iris_ann.ipynb` и `iris.data` в одну папку (например, Colab Notebooks). В локальном Jupyter оставь `iris.data` рядом с ноутбуком.


## 0) Подготовка: библиотеки и настройки

Перед заполнением функций импортируй нужные библиотеки (Pandas, NumPy, инструменты scikit-learn, TensorFlow Keras) и задай глобальные константы.

**Твоя задача:**
- Добавить все импорты в первый кодовый блок.
- Сохранить список имён колонок и вычислить `INPUT_DIM`.
- Настроить `np.random.seed` и `tf.random.set_seed` для воспроизводимости.

**Проверка:**
- Длина `DATA_COLUMNS` равна 5, последний элемент — `'class'`.
- `INPUT_DIM == 4` и константы используются в дальнейших функциях.


In [ ]:
"""This is an example for making an Artificial Neural Network for the iris dataset."""
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

DATA_COLUMNS = ["sepal length", "sepal width", "petal length", "petal width", "class"]
INPUT_DIM = len(DATA_COLUMNS) - 1
TEST_SIZE = 0.3
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


def resolve_iris_data_path(filename="iris.data"):
    """Locate iris.data next to cwd or typical Google Colab Drive folders."""
    candidates = [
        Path(filename),
        Path.cwd() / filename,
        Path("/content/drive/MyDrive/Colab Notebooks") / filename,
        Path("/content/drive/My Drive/Colab Notebooks") / filename,
    ]
    for p in candidates:
        if p.is_file():
            return str(p)
    return str(Path.cwd() / filename)


In [ ]:
def try_to_mount_drive():
    """Mount Google Drive. For local usage only."""
    try:
        from google.colab import drive
        import os

        drive.mount("/content/drive")
        os.chdir("/content/drive/My Drive/Colab Notebooks")
        return True
    except ImportError:
        return False




In [ ]:
# No top-level execution for autograder import safety.
# In notebook, run this manually if needed:
# try_to_mount_drive()


## 1) Загрузка данных

**Твоя задача:**
- Использовать `pd.read_csv`, чтобы загрузить `iris.data`.
- Назначить колонки: `['sepal length', 'sepal width', 'petal length', 'petal width', 'class']`.
- Вернуть DataFrame для следующих шагов.

**Проверка:**
- `df.shape` должен быть `(150, 5)`.
- `df['class'].head()` содержит строковые метки классов.


## 2) Анализ датасета

Перед построением модели полезно проверить распределения признаков и баланс классов.

**Твоя задача:**
- Посмотреть `df.describe()` и другие сводки.
- Проверить баланс классов через `value_counts()`.
- Зафиксировать краткие выводы перед обучением.

**Проверка:**
- Есть отдельные ячейки для анализа и/или заметки.
- Понимание структуры данных сформировано до этапа кодирования и обучения.


In [ ]:
def read_data(filename):
    """Read the iris dataset with the correct column names."""
    path = Path(filename)
    if path.is_file():
        df = pd.read_csv(path, names=DATA_COLUMNS)
        df = df.dropna(how="any").reset_index(drop=True)
        return df

    if path.name == "iris.data":
        from sklearn.datasets import load_iris

        iris = load_iris(as_frame=True)
        X = iris.data.copy()
        X.columns = DATA_COLUMNS[:-1]
        class_map = {i: f"Iris-{name}" for i, name in enumerate(iris.target_names)}
        X["class"] = iris.target.map(class_map)
        return X

    raise FileNotFoundError(f"Dataset file not found: {filename}")


### Обзор датасета (EDA)

Файл UCI может заканчиваться пустой строкой — `read_data` удаляет строки с пропусками.


In [ ]:
def run_eda_overview(filename=None):
    """Run EDA prints manually without top-level side effects."""
    file_to_read = filename or resolve_iris_data_path()
    df = read_data(file_to_read)
    print("форма:", df.shape)
    print(df.head())
    print(df["class"].value_counts())
    print(df.describe())
    return df




In [ ]:
def plot_eda_histograms(df):
    """Plot feature histograms for manual EDA."""
    import matplotlib.pyplot as plt

    feature_cols = DATA_COLUMNS[:-1]
    fig, axes = plt.subplots(2, 2, figsize=(9, 7))
    axes = axes.ravel()
    for ax, col in zip(axes, feature_cols):
        ax.hist(df[col], bins=15, color="steelblue", edgecolor="black", alpha=0.75)
        ax.set_title(col)
    plt.tight_layout()
    plt.show()

    print(
        "Вывод: классы сбалансированы (50+50+50). "
        "Признаки petal разделяют виды лучше, чем sepal — это хорошо для нейросети."
    )




## 3) Кодирование классов в числа

Нейросети нужен числовой target, поэтому колонка `class` преобразуется в целые индексы.

**Твоя задача:**
- Создать `LabelEncoder`, обучить его на `data['class']` и заменить значения.
- Обработать ошибку, если колонки `class` нет (например, `ValueError`).

**Проверка:**
- `sorted(data['class'].unique()) == [0, 1, 2]`.


In [ ]:
def encode_labels_to_numerical(data):
    """Encode the labels to numerical values. eg. iris_color to 0."""
    if "class" not in data.columns:
        raise ValueError("missing required column 'class'")
    le = LabelEncoder()
    data["class"] = le.fit_transform(data["class"].astype(str)).astype(np.int64)
    return data


## 4) Разделение признаков и целевой переменной

**Твоя задача:**
- Использовать `data.iloc[:, :-1]` для признаков.
- Использовать `data.iloc[:, -1]` для меток классов.
- Вернуть `X` и `y`.

**Проверка:**
- `X.shape == (150, 4)` и `y.shape == (150,)`.


In [ ]:
def separate_features_and_labels(data):
    """Split the dataset into features and labels."""
    X = data.iloc[:, :-1]
    y = data.iloc[:, -1]
    return X, y




## 5) One-hot кодирование

Для `categorical_crossentropy` целевые классы должны быть в one-hot формате.

**Твоя задача:**
- Преобразовать `y` в NumPy-массив.
- Использовать `tensorflow.keras.utils.to_categorical`.
- Оставить параметр `num_classes` настраиваемым (по умолчанию 3).

**Проверка:**
- Сумма значений по каждой строке равна 1.
- Форма результата: `(len(y), 3)`.


In [ ]:
def encode_labels_as_one_hot(y, num_classes=3):
    """Encodes labels to one-hot format. Use keras.utils to_categorical function for this."""
    y_arr = np.asarray(y).astype(np.int64)
    return to_categorical(y_arr, num_classes=num_classes).astype(np.float32)


## 6) Разделение на train и test

**Твоя задача:**
- Использовать `train_test_split` с `test_size >= 0.3` и фиксированным `random_state`.
- Применить `stratify`, чтобы сохранить распределение классов.
- Вернуть `X_train, X_test, y_train, y_test`.

**Проверка:**
- Обучающая выборка содержит не менее 70% данных.
- Размеры `X_*` и `y_*` согласованы.


In [ ]:
def split_data_to_test_and_train(X, y):
    """Split the dataset into two for testing and validation."""
    y_labels = np.argmax(np.asarray(y), axis=1)
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_labels,
    )
    return X_train, X_test, y_train, y_test




## 7) Стандартизация признаков

**Твоя задача:**
- Создать `StandardScaler`, обучить его только на `X_train`.
- Применить тот же scaler к `X_test` через `transform`.
- Вернуть масштабированные массивы (желательно `float32`) и scaler.

**Проверка:**
- Среднее `X_train_scaled` близко к 0, стандартное отклонение близко к 1.
- Формы `X_train_scaled` и `X_test_scaled` совпадают с исходными.


In [ ]:
def scale_features(X_train, X_test):
    """Standardize features using the training split as reference."""
    scaler = StandardScaler()
    X_train_np = np.asarray(X_train, dtype=np.float32)
    X_test_np = np.asarray(X_test, dtype=np.float32)
    X_train_scaled = scaler.fit_transform(X_train_np).astype(np.float32)
    X_test_scaled = scaler.transform(X_test_np).astype(np.float32)
    return X_train_scaled, X_test_scaled, scaler




## 8) Архитектура модели

Построй `Sequential` модель: 4 входа и 3 выхода.

**Твоя задача:**
- Добавить как минимум один скрытый слой `Dense` с `relu`.
- Завершить модель выходным `Dense(3, activation='softmax')`.
- Вернуть собранную модель.

**Проверка:**
- `len(model.layers) >= 2`.
- Последний слой имеет `softmax` и `units == 3`.


In [ ]:
def create_model():
    """Create a model with different layers."""
    model = Sequential(
        [
            Input(shape=(INPUT_DIM,)),
            Dense(16, activation="relu"),
            Dense(8, activation="relu"),
            Dense(3, activation="softmax"),
        ]
    )
    return model


## 9) Компиляция модели

Перед обучением модель нужно скомпилировать.

**Твоя задача:**
- Использовать `categorical_crossentropy` как loss.
- Выбрать оптимизатор (например, `adam`) и метрику `accuracy`.
- Вернуть скомпилированную модель.

**Проверка:**
- `model.loss == 'categorical_crossentropy'`.
- Во время обучения отображается `accuracy`.


In [ ]:
def compile_model(model):
    """Compile model with loss, optimizer and metrics."""
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model




## 10) Обучение модели

Обучение выполняется через `model.fit`, который возвращает `History`.

**Твоя задача:**
- Убедиться, что входные данные в формате NumPy и dtype подходит для TensorFlow (`float32`).
- Задать `epochs` и `batch_size`.
- Добавить `validation_data=(X_test, y_test)`.
- Вернуть `history`.

**Проверка:**
- В `history.history` присутствуют `loss` и `val_loss`.
- Обучение завершается корректно и loss в целом снижается.

**Выбор параметров:** 150 эпох, `batch_size=16`, валидация на тестовой части.


In [ ]:
def train_model(model, X_train, X_test, y_train, y_test, epochs=150):
    """Train the model using training and testing datasets."""
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.float32)

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        batch_size=16,
        verbose=0,
    )
    return history




## 11) Оценка модели

Используй встроенный `model.evaluate`, чтобы получить точность на тесте.

**Твоя задача:**
- Привести входные данные к NumPy и корректному dtype.
- Вызвать `model.evaluate` и вернуть точность.
- Не печатать ничего внутри функции.

**Проверка:**
- Возвращаемое значение находится в диапазоне 0...1.


In [ ]:
def evaluate_model(model, X_test, y_test):
    """Evaluate the model and return accuracy."""
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.float32)
    _, evaluation_accuracy = model.evaluate(X_test, y_test, verbose=0)
    return float(evaluation_accuracy)




## 12) Оценка по предсказаниям

Проверим точность вручную через предсказанные классы.

**Твоя задача:**
- Получить предсказания через `model.predict`.
- Преобразовать one-hot в индексы через `np.argmax`.
- Посчитать accuracy через `accuracy_score` или `np.mean`.

**Проверка:**
- Значение близко к результату `model.evaluate`.


In [ ]:
def evaluate_using_predictions(model, X_test, y_test):
    """Evaluate accuracy by comparing predictions and expected labels."""
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.float32)
    probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = np.argmax(y_test, axis=1)
    return float(accuracy_score(y_true, y_pred))




## 13) Главная программа

На этом шаге объединяются все функции в единый пайплайн.

**Твоя задача:**
- Вызвать функции в правильном порядке: загрузка → кодирование → split → масштабирование → модель → обучение → оценка.
- Дополнить `main()` так, чтобы он работал без ручного вмешательства.

**Проверка:**
- `main()` выполняется без ошибок.
- На выходе получаются обе метрики точности.


In [ ]:
def main():
    """Run the full pipeline."""
    try_to_mount_drive()

    np.random.seed(RANDOM_STATE)
    tf.random.set_seed(RANDOM_STATE)
    tf.keras.backend.clear_session()

    iris_file = resolve_iris_data_path()
    data = read_data(iris_file)

    encoded_data = encode_labels_to_numerical(data)
    X, y = separate_features_and_labels(encoded_data)
    y_one_hot = encode_labels_as_one_hot(y, num_classes=3)

    X_train, X_test, y_train, y_test = split_data_to_test_and_train(X, y_one_hot)
    X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

    model = create_model()
    model = compile_model(model)
    history = train_model(model, X_train_scaled, X_test_scaled, y_train, y_test, epochs=150)

    evaluation_accuracy = evaluate_model(model, X_test_scaled, y_test)
    prediction_accuracy = evaluate_using_predictions(model, X_test_scaled, y_test)

    return {
        "evaluation_accuracy": evaluation_accuracy,
        "prediction_accuracy": prediction_accuracy,
        "history": history,
        "model": model,
        "X_test_scaled": X_test_scaled,
        "y_test": y_test,
        "data": data,
        "scaler": scaler,
    }


if __name__ == "__main__":
    results = main()
    print(f"Evaluation accuracy: {results['evaluation_accuracy']:.4f}")
    print(f"Prediction accuracy: {results['prediction_accuracy']:.4f}")


## 14) Динамика обучения и матрица ошибок

Тестовая выборка небольшая (30 процентов); кривые loss показывают, сближаются ли train/validation значения — это визуальная проверка обобщающей способности.


In [ ]:
def show_training_report(results):
    """Show loss curves and confusion matrix manually."""
    import matplotlib.pyplot as plt

    hist = results["history"].history
    plt.figure(figsize=(8, 4))
    plt.plot(hist["loss"], label="train loss")
    plt.plot(hist["val_loss"], label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.title("Потери модели (categorical_crossentropy)")
    plt.tight_layout()
    plt.show()

    y_prob = results["model"].predict(results["X_test_scaled"], verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.argmax(results["y_test"], axis=1)
    print("Матрица ошибок (строки = истинный класс, столбцы = предсказанный):")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4))




## 15) Обсуждение

**Чем использование нейросети отличается от предыдущих методов классификации?**
- В предыдущих заданиях sklearn (например, логистическая регрессия, деревья, KNN) модель обычно проще интерпретируется и строится на более фиксированных правилах.
- Здесь нейросеть обучает многослойные нелинейные представления через градиентный спуск в Keras/TensorFlow (`compile` + `fit`).

**Каковы преимущества и ограничения нейросетей для этого датасета?**
- Преимущества: гибкость нелинейной границы, вероятностный вывод через softmax, быстрое обучение на маленьких данных.
- Ограничения: Iris маленький и относительно простой датасет, поэтому слишком глубокая сеть может переобучаться; интерпретируемость ниже, чем у простых моделей.

**Оправдана ли здесь более простая или более сложная модель?**
- Для Iris обычно достаточно простой архитектуры (1–2 скрытых слоя с умеренным числом нейронов).
- Более сложная сеть редко даёт ощутимый прирост качества, но повышает риск нестабильности и переобучения.
